In [ ]:
# ---
# jupyter:
#   jupytext:
#     text_representation:
#       extension: .py
#       format_name: percent
#       format_version: '1.3'
#       jupytext_version: 1.19.6
#   kernelspec:
#     display_name: Python (latest Jaspy)
#     language: python
#     name: jaspy
# ---

# NB 02 Binary conditional copulas (per fold)

For each blocked fold the divergence w is split at the **fold-training median** into a low and a high regime. In each regime seven parametric families (Gaussian, t, Clayton, Gumbel, Gumbel (180°), Frank, Joe) and a CODINE net (the two per-regime nets of the split CODINE; 3 seeds) are fitted on that fold's training steps of the regime. The pseudo-observations are the **kernel-conditional** ones of notebook 00: the marginals are already conditional on w, and the binary split conditions only the copula. The 49 low/high pairs are built by indexing; the pair with the lowest training AIC is the data-driven binary model, chosen per fold.

Per fold and regime: training AIC, test CVPE inside the regime, marginal-uniformity KS distance D_max (D_max <= 0.07 per regime component; a pair is admissible in a fold only if both of its regime components pass). Each table is written as LaTeX and CSV.

## Imports and per-fold inputs

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"; os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"; os.environ["NUMEXPR_NUM_THREADS"] = "1"
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd

from run_support import load_artifact, save_artifact, SMOKE, MAX_WORKERS   # production defaults; override with THESIS_SMOKE / THESIS_MAX_WORKERS (see run_support.py)
import gate_config as GC
from copula_families import signed_params_of, branch_of, fit_and_evaluate, DEFAULT_FAMILIES   # signed primary parameter; branch (own/turned rotation) of a fit
# Unconditional models / penalty calibration do not use previously calibrated network weights.
os.environ["THESIS_ALLOW_UNCALIBRATED_IMPORT"] = "1"
import fold_pipeline as FP
import reporting_tables
reporting_tables.configure("02_binary_conditional_copulas")

FOLDS = FP.FOLDS
ARR = FP.load_all_folds()          # per-fold arrays: kernel-conditional u1/u2, masks, b, w
print("kernel bandwidth h by fold =", load_artifact("preprocessing_folds_summary").get('kernel_h_by_fold'))

# Fits, marginal KS checks and losses of the 2 x 7 regime families (shared with notebooks 01 and 04 through the
# resume files in artifacts/fold_batch: whichever notebook runs first computes them).
PARAM, _ = FP.run_param_stage(ARR, max_workers=MAX_WORKERS)
# The two regime CODINEs: 4 folds x 3 seeds x 2; marginal KS check on seed 0 only.
NEURAL = FP.run_batch(FP.build_fold_jobs(ARR, None, smoke=SMOKE, kinds=("codine_regime",)),
                      max_workers=MAX_WORKERS, batch_dir=FP.BATCH_DIR, desc='regime CODINE')

## Families per regime and fold

In [ ]:
REGS = ('low', 'high')
RES, TABLES = {}, {}
for f in FOLDS:
    A_ = ARR[f]; tr, te, hi = A_['train'], A_['test'], A_['high']
    RES[f], TABLES[f] = {}, {}
    for reg in REGS:
        rm = hi if reg == 'high' else ~hi
        _, rows = fit_and_evaluate(A_['u1'], A_['u2'], train_mask=tr & rm, test_mask=te & rm,
                                   families=list(DEFAULT_FAMILIES), quiet=True)
        RES[f][reg] = rows
        tab = pd.DataFrame([dict(Family=r['name'], Branch=branch_of(r['cop']), AIC_train=r['aic'], CVPE_test_regime=r['cvpe'], KS_D=PARAM[f].get('ks', {}).get((reg, r['name']), np.nan)) for r in rows])
        tab['Admissible'] = [GC.ks_seed_pass(d) for d in tab.KS_D]
        tab.loc[len(tab)] = dict(Family='CODINE', Branch='-', AIC_train=np.nan, CVPE_test_regime=np.nan, KS_D=max((NEURAL.get(FP.net_key(f, seed, f'{FP.SPLIT_CODINE}|{reg}'), {}).get('ks', np.nan) for seed in FP.SEEDS), default=np.nan), Admissible=all(GC.ks_seed_pass(NEURAL.get(FP.net_key(f, seed, f'{FP.SPLIT_CODINE}|{reg}'), {}).get('ks')) for seed in FP.SEEDS))
        TABLES[f][reg] = tab
    print(f"\n=== fold {f}: train n={int(tr.sum())}, test n={int(te.sum())}; training steps low/high = "
          f"{int((tr & ~hi).sum())}/{int((tr & hi).sum())}; AIC-selected pair (low | high) = {PARAM[f]['selected_pair']}")
    for reg in REGS:
        print(f'{reg} divergence:'); display(TABLES[f][reg])

## Write the per-fold tables

In [ ]:
for f in FOLDS:
    for reg in REGS:
        reporting_tables.write_table(TABLES[f][reg].set_index('Family'), f'binary_copulas_fold_{f}_{reg}',
            f'Binary-split copulas, fold {f}, {reg}-divergence regime: training AIC, CVPE on the regime test steps and marginal-uniformity KS distance (Dmax <= {GC.KS_D_MAX} per regime component).',
            label=f'tab:binary_{f}_{reg}', col_format='llrrrc')

## AIC-selected pair per fold

In [ ]:
sel = pd.DataFrame({f: dict(zip(REGS, PARAM[f]['selected_pair'])) for f in FOLDS})
reporting_tables.show_table(sel, 'aic_selected_pairs', caption='AIC-selected family pair (low, high divergence) per fold.')

## Save

In [ ]:
save_artifact("binary_divergence_copulas_folds",
              tables=TABLES, folds=list(FOLDS), selected_pairs=sel,
              family_params={f: {reg: {r['name']: signed_params_of(r['cop']) for r in RES[f][reg]} for reg in REGS} for f in FOLDS})
print("saved binary_divergence_copulas_folds")

## Table layout

Every table with a tabular is wrapped so that it shrinks to the text width when it is wider and is never enlarged (`reporting_tables.fit_all_tables`); no table needs a manual size edit in the report.

In [ ]:
import reporting_tables
print('table files fitted to the text width:', reporting_tables.fit_all_tables())